In [ ]:
# Импорт необходимых библиотек
import sys
import os
from pathlib import Path

# Добавляем src в путь для импорта модулей
sys.path.append(str(Path.cwd() / "src"))

import torch
import numpy as np
import pandas as pd

# Импорт наших модулей
from config import Config
from load_data import load_data
from tools import build_barrier_labels
from model import create_model, create_optimizer_and_scheduler
from dataset import  BalancedBatchSampler, BalancedBatchBatchSampler, LazyWindowDataset
from torch.utils.data import DataLoader
from trainer import create_trainer

print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA версия: {torch.version.cuda}")
print(f"Устройство: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Создание конфигурации
config = Config.default()

# Настройка параметров для нескольких дней
config.data.train_dates = ["2025-09-22", "2025-09-23", "2025-09-25", "2025-09-26"]  # Список дней для обучения
config.data.val_date = "2025-09-29"                     # День для валидации
config.data.test_date = "2025-09-30"                    # День для тестирования
config.data.data_folder = "./data/npy"

# Параметры модели
config.model.tick_size = 1.0
config.model.theta_ticks = 5
config.model.horizon_sec = 2.0
config.model.window_length = 240
config.model.use_cost_sensitive_focal = True  # Использовать стоимостно-чувствительный лосс

config.training.epochs = 15

print("Конфигурация создана:")
print(f"Дни обучения: {config.data.train_dates}")
print(f"День валидации: {config.data.val_date}")
print(f"День тестирования: {config.data.test_date}")
print(f"Папка с данными: {config.data.data_folder}")
print(f"Эпохи на день: {config.training.epochs}")
print(f"Функция потерь: {'Cost-Sensitive-Focal Loss' if config.model.use_cost_sensitive_focal else 'Focal Loss'}")

In [3]:
# Загрузка и подготовка данных для обучения (MEMORY-EFFICIENT VERSION)
print("Загрузка данных для обучения...")
print("⚡ Используется LazyWindowDataset - экономия памяти!")
train_data_loaders = []

for i, train_date in enumerate(config.data.train_dates):
    print(f"\nОбработка дня обучения {i+1}/{len(config.data.train_dates)}: {train_date}")
    
    # Загрузка данных
    features_file = config.data.get_features_file(train_date)
    print(f"Файл признаков: {features_file}")
    print(f"Папка: {config.data.data_folder}")
    prices_file = config.data.get_prices_file(train_date)
    X, ms, mid = load_data(features_file, prices_file)
    
    # Построение меток
    y_all = build_barrier_labels(
        ms, mid, 
        tick_size=config.model.tick_size,
        theta_ticks=config.model.theta_ticks, 
        horizon_sec=config.model.horizon_sec
    )
    
    # Фильтрация валидных меток
    valid = (y_all != -1)
    Xv, yv = X[valid], y_all[valid]
    
    # LAZY подход - окна создаются на лету, не храним все в памяти
    train_ds = LazyWindowDataset(Xv, yv, config.model.window_length)
    
    # Метки для окон (метка окна = метка последнего элемента)
    y_win = yv[config.model.window_length-1:]
    
    print(f"  Классы и счётчики: {np.unique(y_win, return_counts=True)}")
    print(f"  Датасет: {len(train_ds)} окон | метки: {y_win.shape}")
    print(f"  💾 Память: ~{Xv.nbytes / 1024**3:.2f} ГБ (вместо ~{len(train_ds) * config.model.window_length * Xv.shape[1] * 4 / 1024**3:.2f} ГБ)")
    
    # Создание DataLoader для этого дня
    base_sampler = BalancedBatchSampler(
        y_win, 
        batch_size=config.training.batch_size, 
        num_classes=config.model.num_classes, 
        seed=config.training.seed
    )
    batch_sampler = BalancedBatchBatchSampler(base_sampler, config.training.batch_size)
    train_dl = DataLoader(train_ds, batch_sampler=batch_sampler, num_workers=0)
    train_data_loaders.append(train_dl)

  Классы и счётчики: (array([0, 1, 2], dtype=int64), array([ 18199, 111119,  19208], dtype=int64))
  Датасет: 148526 окон | метки: (148526,)
  💾 Память: ~0.12 ГБ (вместо ~28.02 ГБ)

Обработка дня обучения 4/4: 2025-09-26
Файл признаков: ./data/npy\Si-12.25_2025-09-26_features.npy
Папка: ./data/npy
Загрузка признаков: ./data/npy\Si-12.25_2025-09-26_features.npy
Загрузка цен: ./data/npy\Si-12.25_2025-09-26_prices.csv
Форма массива: (146780, 211)
Тип данных: float32
✓ Данные нормализованы (z-score)
  Диапазон значений: [-341.75, 265.56]
  Классы и счётчики: (array([0, 1, 2], dtype=int64), array([  8330, 128027,   8668], dtype=int64))
  Датасет: 145025 окон | метки: (145025,)
  💾 Память: ~0.11 ГБ (вместо ~27.36 ГБ)


In [4]:
# Загрузка данных для валидации (MEMORY-EFFICIENT VERSION)
print(f"\n⚡ Загрузка данных для валидации: {config.data.val_date}")
val_features_file = config.data.get_features_file(config.data.val_date)
val_prices_file = config.data.get_prices_file(config.data.val_date)
X_val, ms_val, mid_val = load_data(val_features_file, val_prices_file)

y_val_all = build_barrier_labels(
    ms_val, mid_val, 
    tick_size=config.model.tick_size,
    theta_ticks=config.model.theta_ticks, 
    horizon_sec=config.model.horizon_sec
)

valid_val = (y_val_all != -1)
Xv_val, yv_val = X_val[valid_val], y_val_all[valid_val]

# LAZY подход
val_ds = LazyWindowDataset(Xv_val, yv_val, config.model.window_length)
y_win_val = yv_val[config.model.window_length-1:]
val_dl = DataLoader(val_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=0)

print(f"Валидация: {len(val_ds)} окон | метки: {y_win_val.shape}")
print(f"  💾 Память: ~{Xv_val.nbytes / 1024**3:.2f} ГБ")

# Загрузка данных для тестирования (MEMORY-EFFICIENT VERSION)
print(f"\n⚡ Загрузка данных для тестирования: {config.data.test_date}")
test_features_file = config.data.get_features_file(config.data.test_date)
test_prices_file = config.data.get_prices_file(config.data.test_date)
X_test, ms_test, mid_test = load_data(test_features_file, test_prices_file)

y_test_all = build_barrier_labels(
    ms_test, mid_test, 
    tick_size=config.model.tick_size,
    theta_ticks=config.model.theta_ticks, 
    horizon_sec=config.model.horizon_sec
)

valid_test = (y_test_all != -1)
Xv_test, yv_test = X_test[valid_test], y_test_all[valid_test]

# LAZY подход
test_ds = LazyWindowDataset(Xv_test, yv_test, config.model.window_length)
y_win_test = yv_test[config.model.window_length-1:]
test_dl = DataLoader(test_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=0)

print(f"Тестирование: {len(test_ds)} окон | метки: {y_win_test.shape}")
print(f"  💾 Память: ~{Xv_test.nbytes / 1024**3:.2f} ГБ")


⚡ Загрузка данных для валидации: 2025-09-29
Загрузка признаков: ./data/npy\Si-12.25_2025-09-29_features.npy
Загрузка цен: ./data/npy\Si-12.25_2025-09-29_prices.csv
Форма массива: (141587, 211)
Тип данных: float32
✓ Данные нормализованы (z-score)
  Диапазон значений: [-121.87, 106.77]
Валидация: 139811 окон | метки: (139811,)
  💾 Память: ~0.11 ГБ

⚡ Загрузка данных для тестирования: 2025-09-30
Загрузка признаков: ./data/npy\Si-12.25_2025-09-30_features.npy
Загрузка цен: ./data/npy\Si-12.25_2025-09-30_prices.csv
Форма массива: (143169, 211)
Тип данных: float32
✓ Данные нормализованы (z-score)
  Диапазон значений: [-201.36, 314.83]
Тестирование: 141177 окон | метки: (141177,)
  💾 Память: ~0.11 ГБ


In [5]:
print("Все DataLoader'ы созданы успешно!")
print(f"Количество дней обучения: {len(train_data_loaders)}")
print(f"Валидационный DataLoader: {len(val_dl)} батчей")
print(f"Тестовый DataLoader: {len(test_dl)} батчей")


Все DataLoader'ы созданы успешно!
Количество дней обучения: 4
Валидационный DataLoader: 274 батчей
Тестовый DataLoader: 276 батчей


In [6]:
# Создание модели
model, criterion = create_model(config.model)
optimizer, scheduler, scaler = create_optimizer_and_scheduler(model, config.training)

model = model.to(config.training.device)

print(f"Модель создана: {sum(p.numel() for p in model.parameters())} параметров")
print(f"Устройство: {config.training.device}")


🔧 Создание улучшенной модели:
   input_features: 211
   hidden_size: 96
   num_classes: 3
   Residual connections: ✓
   Dilations: [1, 2, 4, 8, 16]
   Attention mechanism: ✓
📉 Используется Weighted CrossEntropyLoss
   Веса классов: [10.0, 1.0, 10.0]
   Label smoothing: 0.1
Модель создана: 535395 параметров
Устройство: cuda


In [7]:
# ДОБАВЬТЕ ЭТУ СТРОКУ:
config.print_config()


📋 КОНФИГУРАЦИЯ ОБУЧЕНИЯ

🗂️  ДАННЫЕ:
   Дни обучения: ['2025-09-22', '2025-09-23', '2025-09-25', '2025-09-26']
   День валидации: 2025-09-29
   День тестирования: 2025-09-30
   Папка данных: ./data/npy

🏗️  АРХИТЕКТУРА МОДЕЛИ:
   input_features: 211
   hidden_size: 96
   num_classes: 3
   groups: 8
   dropout: 0.5
   window_length: 240

📊 LOSS FUNCTION:
   use_cost_sensitive_focal: True
   focal_alpha: [10.0, 1.0, 10.0]
   focal_gamma: 2.5
   focal_alpha_weight: 0.25
   cost_weight: 1.0
   label_smoothing: 0.1

⚙️  ОПТИМИЗАЦИЯ:
   batch_size: 512
   learning_rate: 0.0001
   weight_decay: 0.0005
   epochs: 15
   grad_clip_norm: 0.3
   scheduler_type: cosine
   device: cuda




In [8]:
# Создание тренера и последовательное обучение
trainer = create_trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=config.training.device,
    grad_clip_norm=config.training.grad_clip_norm
)

# Последовательное обучение на нескольких днях
print("Начало последовательного обучения...")
training_results = trainer.train_sequential_days(
    train_data_loaders=train_data_loaders,
    val_loader=val_dl,
    epochs_per_day=config.training.epochs,
    save_path_template="model_day_{}.pt",
    verbose=True
)

print(f"\nЛучший macro-F1 на валидации: {training_results['best_score']:.4f}")


Начало последовательного обучения...

[2025-10-07 09:54:15] ОБУЧЕНИЕ НА ДНЕ 1/4

🚀 ИНФОРМАЦИЯ ОБ ОБУЧЕНИИ

📊 МОДЕЛЬ:
   Всего параметров: 535,395
   Обучаемых параметров: 535,395
   Размер модели: ~2.04 МБ (float32)

📦 ДАННЫЕ:
   Train батчей: 277
   Val батчей: 274
   Размер батча: torch.Size([512, 211, 240])
   Тип данных: torch.float32
   Устройство: cpu
   Классы в батче: [0, 1, 2]
   Распределение: [171, 171, 170]

⚙️  ОПТИМИЗАТОР:
   Тип: AdamW
   Learning rate: 1.00e-04
   Weight decay: 5.00e-04
   Scheduler: CosineAnnealingLR
   Gradient clipping: 0.3

📉 LOSS FUNCTION:
   Тип: WeightedCrossEntropyLoss
   label_smoothing: 0.1

🎯 ОБУЧЕНИЕ:
   Эпох: 15
   Устройство: cuda
   Mixed precision: True

▶️  НАЧАЛО ОБУЧЕНИЯ...

[09:55:58] [01] train_loss=0.8643 acc=0.385 | val_loss=2.7159 macroF1=0.068 F1↓=0.097 F1○=0.000 F1↑=0.107 (103.2s)
Confusion matrix [rows=true, cols=pred]:
[[ 3591     0  3818]
 [60107     0 64792]
 [ 3200     0  4303]]

[09:57:39] [02] train_loss=0.6637 acc=0.543

RuntimeError: unscale_() has already been called on this optimizer since the last update().

In [ ]:
# ============================================================
# ЗАГРУЗКА ОБУЧЕННОЙ МОДЕЛИ ИЗ ФАЙЛА
# ============================================================

# Укажите имя файла модели для загрузки
model_file = "model_day_1.pt"  # Можно изменить на любой файл: "model_day_2.pt", "best_deeplob_like.pt" и т.д.

print(f"📥 Загрузка модели из файла: {model_file}")
print(f"   Путь: {os.path.abspath(model_file)}")

if os.path.exists(model_file):
    # Загрузка весов модели
    state_dict = torch.load(model_file, map_location=config.training.device)
    model.load_state_dict(state_dict)
    
    print("✅ Модель успешно загружена!")
    print(f"   Количество параметров: {sum(p.numel() for p in model.parameters()):,}")
    print(f"   Устройство: {next(model.parameters()).device}")
    
    # Проверка модели на валидационных данных
    print("\n🔍 Проверка загруженной модели на валидационном наборе...")
    val_metrics = trainer.evaluate(val_dl)
    
    print(f"\n📊 Метрики на валидации:")
    print(f"   Val Loss: {val_metrics['loss']:.4f}")
    print(f"   Macro F1: {val_metrics['macro_f1']:.4f}")
    print(f"   F1 ↓ (down): {val_metrics['f1_down']:.3f}")
    print(f"   F1 ○ (flat): {val_metrics['f1_flat']:.3f}")
    print(f"   F1 ↑ (up):   {val_metrics['f1_up']:.3f}")
    
    print("\n📈 Confusion Matrix:")
    print(val_metrics['confusion_matrix'])
    
else:
    print(f"❌ ОШИБКА: Файл '{model_file}' не найден!")
    print(f"   Убедитесь, что файл существует в текущей директории.")
    print(f"   Доступные файлы моделей:")
    model_files = [f for f in os.listdir('.') if f.endswith('.pt')]
    if model_files:
        for f in sorted(model_files):
            print(f"      - {f}")
    else:
        print("      (нет файлов .pt в текущей директории)")


In [ ]:
# Тестирование и сохранение модели
test_results = trainer.test(test_dl)

# Сохранение финальной модели
model_path = "best_deeplob_like.pt"
trainer.save_model(model_path)

print(f"\nОбучение завершено!")
print(f"Финальная модель сохранена: {os.path.abspath(model_path)}")
print(f"Промежуточные модели сохранены: model_day_1.pt, model_day_2.pt, ...")
